In [1]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb

horizons = [1, 3, 6, 12, 24]
test_start = pd.Timestamp("2026-08-01")

def to_naive_utc(s):
    s = pd.to_datetime(s)
    if s.dt.tz is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s

rmse = lambda a, b: np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

# PM2.5 lookup by (station, time), used for diurnal persistence
merged = pd.read_csv("../data/processed/kathmandu_merged.csv")
merged["datetime_utc"] = to_naive_utc(merged["datetime_utc"])
merged["station"] = merged["station"].map({"teku": "Teku", "mid_baneshwor": "Mid Baneshwor"})
pm = merged.set_index(["station", "datetime_utc"])["pm25"]

def lookup(stations, times):
    return pm.reindex(pd.MultiIndex.from_arrays([stations, times])).values

In [2]:
# Diurnal persistence: forecast y(t+h) with the value 24 hours before the target, y(t+h-24)
rows = []
for h in horizons:
    # Hold-out
    ho = pd.read_csv(f"../models/holdout_predictions_{h}h.csv")
    ho["time"] = to_naive_utc(ho["time"])
    ho["diurnal"] = lookup(ho["station"], ho["time"] + pd.Timedelta(hours=h - 24))
    ho = ho.dropna(subset=["diurnal"])

    # CV (mean of per-fold RMSE)
    cv = pd.read_csv(f"../models/cv_predictions_{h}h.csv")
    cv["time"] = to_naive_utc(cv["time"])
    cv["current"] = lookup(cv["station"], cv["time"])
    cv["diurnal"] = lookup(cv["station"], cv["time"] + pd.Timedelta(hours=h - 24))
    cv = cv.dropna(subset=["current", "diurnal"])
    folds = cv.groupby("fold").apply(lambda g: pd.Series({
        "model": rmse(g["predicted"], g["actual"]),
        "persistence": rmse(g["current"], g["actual"]),
        "diurnal": rmse(g["diurnal"], g["actual"]),
    }))

    rows.append({
        "horizon": h,
        "cv_model": folds["model"].mean(),
        "cv_persistence": folds["persistence"].mean(),
        "cv_diurnal": folds["diurnal"].mean(),
        "ho_rows": len(ho),
        "ho_model": rmse(ho["predicted"], ho["actual"]),
        "ho_persistence": rmse(ho["current_pm25"], ho["actual"]),
        "ho_diurnal": rmse(ho["diurnal"], ho["actual"]),
    })

diurnal_results = pd.DataFrame(rows).round(2)
diurnal_results.to_csv("../models/diurnal_baseline_results.csv", index=False)
diurnal_results

C:\Users\khush\AppData\Local\Temp\ipykernel_51472\4293731681.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  folds = cv.groupby("fold").apply(lambda g: pd.Series({
C:\Users\khush\AppData\Local\Temp\ipykernel_51472\4293731681.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  folds = cv.groupby("fold").apply(lambda g: pd.Series({
C:\Users\khush\AppData\Local\Temp\ipykernel_51472\4293731681.py:16: FutureWarning:

,horizon,cv_model,cv_persistence,cv_diurnal,ho_rows,ho_model,ho_persistence,ho_diurnal
0,1,14.78,17.00,31.06,1325,6.57,7.05,11.03
1,3,22.18,31.64,30.97,1321,9.51,11.94,11.02
2,6,25.32,41.87,30.84,1315,9.73,14.96,10.99
3,12,27.31,35.37,30.58,1308,12.14,12.53,11.03
4,24,28.06,30.46,30.46,1302,11.32,11.06,11.06


In [4]:
df = pd.read_csv("../data/processed/kathmandu_features.csv")
df["datetime_utc"] = to_naive_utc(df["datetime_utc"])
df = df.sort_values("datetime_utc").reset_index(drop=True)

target_columns = [c for c in df.columns if c.startswith("target_")]
feature_columns = [c for c in df.columns if c not in ["datetime_utc"] + target_columns]

rows = []
for h in horizons:
    target = df[f"target_pm25_next_{h}h"]
    valid = target.notna() & df["pm25"].notna()
    d = df[valid]
    X, y_abs = d[feature_columns], target[valid]
    y_delta = y_abs - d["pm25"]

    dev = (d["datetime_utc"] < test_start - pd.Timedelta(hours=h)).values
    hold = (d["datetime_utc"] >= test_start).values

    model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3,
                             random_state=42, n_jobs=-1)
    model.fit(X[dev], y_delta[dev])
    pred = d["pm25"][hold] + model.predict(X[hold])

    with open(f"../models/tuning_results_{h}h.json") as f:
        tuned = json.load(f)

    rows.append({
        "horizon": h,
        "untuned_holdout_rmse": rmse(pred, y_abs[hold]),
        "tuned_holdout_rmse": tuned.get("holdout_rmse"),
        "persistence_holdout_rmse": rmse(d["pm25"][hold], y_abs[hold]),
    })

untuned_results = pd.DataFrame(rows).round(2)
untuned_results.to_csv("../models/untuned_holdout_results.csv", index=False)
untuned_results

,horizon,untuned_holdout_rmse,tuned_holdout_rmse,persistence_holdout_rmse
0,1,6.54,6.56,7.05
1,3,9.90,9.53,12.01
2,6,10.55,9.76,15.02
3,12,11.85,12.14,12.53
4,24,10.75,11.31,11.06
